# Irrigation Dataset Pre-Training Analysis

This notebook performs pre-training analysis for the irrigation segmentation dataset across **Colorado** and **Utah**.
It is designed to run on CPU-only login nodes and focuses on:

1. Class imbalance and recommended class weights
2. Spectral separability between irrigation classes
3. Temporal signatures across seasons (`s3`, `s4`, `s5`)
4. Label noise assessment using NDVI heuristics
5. Cross-state distribution shift (OOD difficulty)
6. Spatial distribution of labels
7. Representative qualitative samples
8. Data-driven modeling recommendations

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm

try:
    from scipy.stats import wasserstein_distance
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
RNG = np.random.default_rng(42)

## Section 0 — Configuration and Data Loading Helpers
Set data paths and reusable utilities.

In [ ]:
DATA_ROOT = Path('/path/to/data_root')  # <-- update this
MAX_SPECTRAL_TILES_PER_STATE = 2000
PIXELS_PER_CLASS_PER_TILE = 100
SCATTER_POINTS_PER_CLASS = 30000

BAND_NAMES = [
    "B02_Blue", "B03_Green", "B04_Red", "B08_NIR",
    "B8A_NarrowNIR", "B11_SWIR1_median", "B12_SWIR2_median",
    "B11_SWIR1_min", "B12_SWIR2_min",
    "NDVI_median", "NDVI_max", "NDMI_median", "NDMI_max",
    "valid_count"
]
SEASONS = ["s3", "s4", "s5"]
CLASS_NAMES = {0: "Background", 1: "Flood", 2: "Sprinkler", 3: "Drip"}
CLASS_COLORS = {0: "#808080", 1: "#2196F3", 2: "#4CAF50", 3: "#FF9800"}
IRRIGATED_CLASSES = [1, 2, 3]

STATE_PATHS = {
    'Colorado': DATA_ROOT / 'Colorado',
    'Utah': DATA_ROOT / 'Utah'
}

for state, path in STATE_PATHS.items():
    print(f"{state}: {path} {'✅' if path.exists() else '❌ (missing)'}")

In [ ]:
def load_tile(state_path: Path, tile_id: int, season: str = 's4'):
    """Load image and label for a single tile."""
    tile_name = f"tile_{tile_id:04d}"
    img_path = state_path / 'images' / f"{tile_name}_{season}.tif"
    lbl_path = state_path / 'labels' / f"{tile_name}_label.tif"
    with rasterio.open(img_path) as src:
        image = src.read()
    with rasterio.open(lbl_path) as src:
        label = src.read(1)
    return image, label


def get_tile_ids(state_path: Path):
    ids = []
    for f in sorted((state_path / 'labels').glob('tile_*_label.tif')):
        tid = int(f.stem.split('_')[1])
        ids.append(tid)
    return ids


def sample_tile_ids(tile_ids, max_tiles, rng=RNG):
    if len(tile_ids) <= max_tiles:
        return list(tile_ids)
    return sorted(rng.choice(tile_ids, size=max_tiles, replace=False).tolist())


def choose_dominant_irrigation_class(label):
    vals, counts = np.unique(label, return_counts=True)
    class_counts = dict(zip(vals.tolist(), counts.tolist()))
    irrigated = {c: class_counts.get(c, 0) for c in IRRIGATED_CLASSES}
    if sum(irrigated.values()) == 0:
        return 0
    return max(irrigated, key=irrigated.get)


def robust_normalize(img, lo=2, hi=98):
    out = np.zeros_like(img, dtype=np.float32)
    for b in range(img.shape[0]):
        band = img[b]
        p_lo, p_hi = np.nanpercentile(band, [lo, hi])
        out[b] = 0 if p_hi <= p_lo else np.clip((band - p_lo) / (p_hi - p_lo), 0, 1)
    return out


def bhattacharyya_distance_from_hist(x, y, bins=80):
    x = np.asarray(x)
    y = np.asarray(y)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    lo = min(x.min(), y.min())
    hi = max(x.max(), y.max())
    if hi <= lo:
        return 0.0
    px, edges = np.histogram(x, bins=bins, range=(lo, hi), density=True)
    py, _ = np.histogram(y, bins=edges, density=True)
    px = px + 1e-12
    py = py + 1e-12
    bc = np.sum(np.sqrt(px * py)) * np.diff(edges).mean()
    return float(-np.log(np.clip(bc, 1e-12, 1.0)))

### Collect tile indices

In [ ]:
all_tile_ids = {state: get_tile_ids(path) for state, path in STATE_PATHS.items()}
for state, ids in all_tile_ids.items():
    print(f"{state}: {len(ids):,} tiles")

sample_tile_ids_by_state = {
    state: sample_tile_ids(ids, MAX_SPECTRAL_TILES_PER_STATE)
    for state, ids in all_tile_ids.items()
}
for state, ids in sample_tile_ids_by_state.items():
    print(f"{state} sample for spectral/temporal/noise/OOD analyses: {len(ids):,} tiles")

## Section 1 — Class Distribution Analysis
Compute pixel-level and tile-level class imbalance statistics across all labels.

In [ ]:
pixel_counts_by_state = {}
tile_presence_by_state = {}

for state, state_path in STATE_PATHS.items():
    pixel_counter = Counter()
    tile_counter = Counter()
    for tid in tqdm(all_tile_ids[state], desc=f"Counting labels ({state})"):
        lbl_path = state_path / 'labels' / f"tile_{tid:04d}_label.tif"
        with rasterio.open(lbl_path) as src:
            label = src.read(1)

        vals, counts = np.unique(label, return_counts=True)
        present = set(vals.tolist())
        for v, c in zip(vals.tolist(), counts.tolist()):
            pixel_counter[int(v)] += int(c)
        for cls in range(4):
            if cls in present:
                tile_counter[cls] += 1

    pixel_counts_by_state[state] = pixel_counter
    tile_presence_by_state[state] = tile_counter

In [ ]:
rows = []
for state in STATE_PATHS:
    total_pixels = sum(pixel_counts_by_state[state].values())
    for cls in range(4):
        count = pixel_counts_by_state[state][cls]
        rows.append({
            'state': state,
            'class_id': cls,
            'class_name': CLASS_NAMES[cls],
            'pixel_count': count,
            'pixel_pct': count / total_pixels * 100 if total_pixels else 0.0,
            'tiles_with_class': tile_presence_by_state[state][cls],
            'tile_pct': tile_presence_by_state[state][cls] / len(all_tile_ids[state]) * 100 if all_tile_ids[state] else 0.0,
        })

class_dist_df = pd.DataFrame(rows)
global_counts = class_dist_df.groupby('class_id')['pixel_count'].sum().reindex(range(4)).fillna(0).values
safe_counts = np.where(global_counts == 0, 1, global_counts)
inv = 1.0 / safe_counts
weights = inv / inv.mean()
weight_map = {int(i): float(w) for i, w in enumerate(weights)}
class_dist_df['recommended_weight_global'] = class_dist_df['class_id'].map(weight_map)

display(class_dist_df.sort_values(['state', 'class_id']))
print('Recommended class weights (global inverse-frequency, normalized):')
for cls in range(4):
    print(f"  {cls} ({CLASS_NAMES[cls]}): {weight_map[cls]:.4f}")

for state in STATE_PATHS:
    n_tiles = len(all_tile_ids[state])
    flood = tile_presence_by_state[state][1] / n_tiles * 100 if n_tiles else 0
    sprinkler = tile_presence_by_state[state][2] / n_tiles * 100 if n_tiles else 0
    drip = tile_presence_by_state[state][3] / n_tiles * 100 if n_tiles else 0
    print(f"{state}: {flood:.2f}% tiles contain flood, {sprinkler:.2f}% contain sprinkler, {drip:.2f}% contain drip")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, state in zip(axes, STATE_PATHS.keys()):
    sub = class_dist_df[class_dist_df['state'] == state].sort_values('class_id')
    ax.bar(sub['class_name'], sub['pixel_count'], color=[CLASS_COLORS[c] for c in sub['class_id']])
    ax.set_title(f'{state}: Pixel Count per Class')
    ax.set_xlabel('Class')
    ax.set_ylabel('Pixel count')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

**Interpretation placeholder:**
- Fill in imbalance severity and any decision about retaining/merging rare classes (especially drip) based on computed percentages.

## Section 2 — Spectral Separability Analysis
Estimate per-band class separability using sampled pixels from Colorado.

In [ ]:
co_path = STATE_PATHS['Colorado']
co_ids = sample_tile_ids_by_state['Colorado']
band_values = {cls: defaultdict(list) for cls in IRRIGATED_CLASSES}

for tid in tqdm(co_ids, desc='Collecting spectral samples (Colorado)'):
    image, label = load_tile(co_path, tid, season='s4')
    for cls in IRRIGATED_CLASSES:
        ys, xs = np.where(label == cls)
        if len(ys) == 0:
            continue
        k = min(PIXELS_PER_CLASS_PER_TILE, len(ys))
        idx = RNG.choice(len(ys), size=k, replace=False)
        ys_sel, xs_sel = ys[idx], xs[idx]
        vals = image[:, ys_sel, xs_sel]
        for b in range(vals.shape[0]):
            band_values[cls][b].append(vals[b])

for cls in IRRIGATED_CLASSES:
    for b in range(len(BAND_NAMES)):
        band_values[cls][b] = np.concatenate(band_values[cls][b]) if band_values[cls][b] else np.array([])

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()
for b in range(len(BAND_NAMES)):
    ax = axes[b]
    for cls in IRRIGATED_CLASSES:
        vals = band_values[cls][b]
        if len(vals) < 10:
            continue
        sns.kdeplot(vals, ax=ax, label=CLASS_NAMES[cls], color=CLASS_COLORS[cls], fill=False, linewidth=1.2, warn_singular=False)
    ax.set_title(BAND_NAMES[b])
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
for i in [14, 15]:
    axes[i].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
sep_rows = []
for b, band_name in enumerate(BAND_NAMES):
    flood = band_values[1][b]
    sprinkler = band_values[2][b]
    if len(flood) < 10 or len(sprinkler) < 10:
        sep_rows.append({'band_idx': b, 'band_name': band_name, 'bhattacharyya_flood_vs_sprinkler': np.nan, 'abs_mean_gap': np.nan})
        continue
    sep_rows.append({
        'band_idx': b,
        'band_name': band_name,
        'bhattacharyya_flood_vs_sprinkler': bhattacharyya_distance_from_hist(flood, sprinkler, bins=100),
        'abs_mean_gap': abs(np.nanmean(flood) - np.nanmean(sprinkler)),
    })
sep_df = pd.DataFrame(sep_rows).sort_values('bhattacharyya_flood_vs_sprinkler', ascending=False)
display(sep_df)
print('Top 5 bands by Flood vs Sprinkler Bhattacharyya distance:')
print(sep_df[['band_name', 'bhattacharyya_flood_vs_sprinkler']].head(5).to_string(index=False))

**Interpretation placeholder:**
- Identify which bands most separate flood vs sprinkler (often SWIR/ND* features) and whether RGB overlap is high.

## Section 3 — Temporal Signature Analysis
Compare seasonal trajectories for key bands by class (mean ± std).

In [ ]:
KEY_BANDS = {'NDVI_median': 9, 'NDMI_median': 11, 'B11_SWIR1_min': 7}
temporal_values = {name: {cls: {s: [] for s in SEASONS} for cls in IRRIGATED_CLASSES} for name in KEY_BANDS}

for tid in tqdm(co_ids, desc='Collecting temporal samples (Colorado)'):
    season_images = {}
    labels = None
    for s in SEASONS:
        img, lbl = load_tile(co_path, tid, season=s)
        season_images[s] = img
        labels = lbl
    for cls in IRRIGATED_CLASSES:
        ys, xs = np.where(labels == cls)
        if len(ys) == 0:
            continue
        k = min(PIXELS_PER_CLASS_PER_TILE, len(ys))
        idx = RNG.choice(len(ys), size=k, replace=False)
        ys_sel, xs_sel = ys[idx], xs[idx]
        for band_name, bidx in KEY_BANDS.items():
            for s in SEASONS:
                temporal_values[band_name][cls][s].append(season_images[s][bidx, ys_sel, xs_sel])

for band_name in KEY_BANDS:
    for cls in IRRIGATED_CLASSES:
        for s in SEASONS:
            temporal_values[band_name][cls][s] = np.concatenate(temporal_values[band_name][cls][s]) if temporal_values[band_name][cls][s] else np.array([])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
season_idx = np.arange(len(SEASONS))
for ax, band_name in zip(axes, KEY_BANDS.keys()):
    for cls in IRRIGATED_CLASSES:
        means, stds = [], []
        for s in SEASONS:
            vals = temporal_values[band_name][cls][s]
            means.append(np.nanmean(vals) if len(vals) else np.nan)
            stds.append(np.nanstd(vals) if len(vals) else np.nan)
        means = np.array(means, dtype=float)
        stds = np.array(stds, dtype=float)
        ax.plot(season_idx, means, marker='o', label=CLASS_NAMES[cls], color=CLASS_COLORS[cls])
        ax.fill_between(season_idx, means - stds, means + stds, alpha=0.15, color=CLASS_COLORS[cls])
    ax.set_title(f'Temporal profile: {band_name}')
    ax.set_xticks(season_idx)
    ax.set_xticklabels(SEASONS)
    ax.set_xlabel('Season')
    ax.set_ylabel('Value')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**Interpretation placeholder:**
- Describe class-specific seasonal behavior and whether temporal input likely improves discrimination.

## Section 4 — Label Noise Assessment
Use NDVI thresholds as a proxy for likely false positives/false negatives in labels.

In [ ]:
NDVI_IDX = 9
NDMI_IDX = 11
LOW_NDVI_THR = 0.15
HIGH_NDVI_THR = 0.40

ndvi_bg, ndvi_irr = [], []
scatter_by_class = {c: {'ndvi': [], 'ndmi': []} for c in range(4)}
irr_low_count = irr_total = bg_high_count = bg_total = 0

for tid in tqdm(co_ids, desc='Assessing label noise (Colorado s4)'):
    image, label = load_tile(co_path, tid, season='s4')
    ndvi = image[NDVI_IDX]
    ndmi = image[NDMI_IDX]

    irr_mask = np.isin(label, IRRIGATED_CLASSES)
    bg_mask = label == 0
    irr_vals = ndvi[irr_mask]
    bg_vals = ndvi[bg_mask]

    ndvi_irr.append(irr_vals)
    ndvi_bg.append(bg_vals)
    irr_total += irr_vals.size
    bg_total += bg_vals.size
    irr_low_count += int(np.sum(irr_vals < LOW_NDVI_THR))
    bg_high_count += int(np.sum(bg_vals > HIGH_NDVI_THR))

    for cls in range(4):
        ys, xs = np.where(label == cls)
        if len(ys) == 0:
            continue
        k = min(PIXELS_PER_CLASS_PER_TILE, len(ys))
        idx = RNG.choice(len(ys), size=k, replace=False)
        ys_sel, xs_sel = ys[idx], xs[idx]
        scatter_by_class[cls]['ndvi'].append(ndvi[ys_sel, xs_sel])
        scatter_by_class[cls]['ndmi'].append(ndmi[ys_sel, xs_sel])

ndvi_bg = np.concatenate(ndvi_bg) if ndvi_bg else np.array([])
ndvi_irr = np.concatenate(ndvi_irr) if ndvi_irr else np.array([])
for cls in range(4):
    scatter_by_class[cls]['ndvi'] = np.concatenate(scatter_by_class[cls]['ndvi']) if scatter_by_class[cls]['ndvi'] else np.array([])
    scatter_by_class[cls]['ndmi'] = np.concatenate(scatter_by_class[cls]['ndmi']) if scatter_by_class[cls]['ndmi'] else np.array([])

pct_irr_low = (irr_low_count / irr_total * 100) if irr_total else np.nan
pct_bg_high = (bg_high_count / bg_total * 100) if bg_total else np.nan
print(f"{pct_irr_low:.2f}% of irrigated pixels have NDVI < {LOW_NDVI_THR} (possible false positives / fallow)")
print(f"{pct_bg_high:.2f}% of background pixels have NDVI > {HIGH_NDVI_THR} (possible false negatives / unlabeled irrigated)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
if len(ndvi_bg):
    sns.histplot(ndvi_bg, bins=120, stat='density', color=CLASS_COLORS[0], label='Background', alpha=0.35, ax=ax)
if len(ndvi_irr):
    sns.histplot(ndvi_irr, bins=120, stat='density', color='#7E57C2', label='All Irrigated (1/2/3)', alpha=0.35, ax=ax)
ax.axvline(LOW_NDVI_THR, color='red', linestyle='--', linewidth=1.2, label=f'Low threshold ({LOW_NDVI_THR})')
ax.axvline(HIGH_NDVI_THR, color='darkred', linestyle='--', linewidth=1.2, label=f'High threshold ({HIGH_NDVI_THR})')
ax.set_title('NDVI distribution: Background vs Irrigated')
ax.set_xlabel('NDVI_median')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
for cls in range(4):
    ndvi_vals = scatter_by_class[cls]['ndvi']
    ndmi_vals = scatter_by_class[cls]['ndmi']
    if len(ndvi_vals) == 0:
        continue
    n = min(SCATTER_POINTS_PER_CLASS, len(ndvi_vals))
    idx = RNG.choice(len(ndvi_vals), size=n, replace=False)
    ax.scatter(ndvi_vals[idx], ndmi_vals[idx], s=4, alpha=0.25, color=CLASS_COLORS[cls], label=CLASS_NAMES[cls])
ax.set_title('NDVI vs NDMI by class (sampled pixels)')
ax.set_xlabel('NDVI_median')
ax.set_ylabel('NDMI_median')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

**Interpretation placeholder:**
- Quantify apparent noise rates and decide whether bidirectional NDVI-based cleaning thresholds are justified.

## Section 5 — Cross-State Distribution Shift
Compare class-conditional distributions between Colorado and Utah and quantify shift magnitude.

In [ ]:
OOD_BANDS = {'NDVI_median': 9, 'NDMI_median': 11, 'B11_SWIR1_min': 7, 'B11_SWIR1_median': 5}
state_class_band_values = {
    state: {cls: {name: [] for name in OOD_BANDS} for cls in IRRIGATED_CLASSES}
    for state in STATE_PATHS
}

for state, state_path in STATE_PATHS.items():
    ids = sample_tile_ids_by_state[state]
    for tid in tqdm(ids, desc=f'Collecting OOD samples ({state})'):
        image, label = load_tile(state_path, tid, season='s4')
        for cls in IRRIGATED_CLASSES:
            ys, xs = np.where(label == cls)
            if len(ys) == 0:
                continue
            k = min(PIXELS_PER_CLASS_PER_TILE, len(ys))
            idx = RNG.choice(len(ys), size=k, replace=False)
            ys_sel, xs_sel = ys[idx], xs[idx]
            for band_name, bidx in OOD_BANDS.items():
                state_class_band_values[state][cls][band_name].append(image[bidx, ys_sel, xs_sel])

for state in STATE_PATHS:
    for cls in IRRIGATED_CLASSES:
        for band_name in OOD_BANDS:
            state_class_band_values[state][cls][band_name] = np.concatenate(state_class_band_values[state][cls][band_name]) if state_class_band_values[state][cls][band_name] else np.array([])

In [ ]:
fig, axes = plt.subplots(len(IRRIGATED_CLASSES), len(OOD_BANDS), figsize=(14, 9), sharex=False, sharey=False)
if axes.ndim == 1:
    axes = axes[None, :]

for r, cls in enumerate(IRRIGATED_CLASSES):
    for c, band_name in enumerate(OOD_BANDS.keys()):
        ax = axes[r, c]
        co_vals = state_class_band_values['Colorado'][cls][band_name]
        ut_vals = state_class_band_values['Utah'][cls][band_name]
        if len(co_vals) > 10:
            sns.kdeplot(co_vals, ax=ax, color='#1E88E5', label='Colorado', linewidth=1.2, warn_singular=False)
        if len(ut_vals) > 10:
            sns.kdeplot(ut_vals, ax=ax, color='#D81B60', label='Utah', linewidth=1.2, warn_singular=False)
        if r == 0:
            ax.set_title(band_name)
        if c == 0:
            ax.set_ylabel(CLASS_NAMES[cls])
        if r == 0 and c == 0:
            ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def hist_l1_distance(x, y, bins=80):
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    if hi <= lo:
        return 0.0
    px, edges = np.histogram(x, bins=bins, range=(lo, hi), density=True)
    py, _ = np.histogram(y, bins=edges, density=True)
    px = px / (px.sum() + 1e-12)
    py = py / (py.sum() + 1e-12)
    return float(np.sum(np.abs(px - py)))

shift_rows = []
for cls in IRRIGATED_CLASSES:
    for band_name in OOD_BANDS:
        co_vals = state_class_band_values['Colorado'][cls][band_name]
        ut_vals = state_class_band_values['Utah'][cls][band_name]
        if HAS_SCIPY and len(co_vals) > 10 and len(ut_vals) > 10:
            shift = float(wasserstein_distance(co_vals, ut_vals))
            metric = 'wasserstein'
        else:
            shift = hist_l1_distance(co_vals, ut_vals)
            metric = 'hist_l1'
        shift_rows.append({'class_id': cls, 'class_name': CLASS_NAMES[cls], 'band_name': band_name, 'shift_metric': metric, 'shift_value': shift})

shift_df = pd.DataFrame(shift_rows)
display(shift_df)

pivot = shift_df.pivot(index='class_name', columns='band_name', values='shift_value')
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='magma', ax=ax)
ax.set_title('Cross-state distribution shift magnitude (Colorado vs Utah)')
plt.tight_layout()
plt.show()

**Interpretation placeholder:**
- List most shifted and most stable features, and implications for OOD generalization strategy.

## Section 6 — Spatial Distribution of Labels
Map dominant irrigation class per tile center using metadata coordinates.

In [ ]:
def load_metadata_with_dominant_class(state, state_path):
    meta = pd.read_csv(state_path / 'metadata.csv')
    tile_col_candidates = ['tile_id', 'id', 'tile', 'tile_index']
    tile_col = next((c for c in tile_col_candidates if c in meta.columns), None)
    if tile_col is None:
        str_cols = [c for c in meta.columns if meta[c].dtype == 'object']
        found = None
        for c in str_cols:
            if meta[c].astype(str).str.contains('tile_').any():
                found = c
                break
        if found is None:
            raise ValueError(f"Could not infer tile id column in {state_path / 'metadata.csv'}")
        meta['tile_id_inferred'] = meta[found].astype(str).str.extract(r'(\d+)').astype(int)
        tile_col = 'tile_id_inferred'

    dominant = {}
    for tid in tqdm(all_tile_ids[state], desc=f'Dominant class from labels ({state})'):
        with rasterio.open(state_path / 'labels' / f"tile_{tid:04d}_label.tif") as src:
            label = src.read(1)
        dominant[tid] = choose_dominant_irrigation_class(label)

    meta = meta.copy()
    meta['tile_id'] = meta[tile_col].astype(int)
    meta['dominant_class'] = meta['tile_id'].map(dominant).fillna(0).astype(int)
    meta['dominant_class_name'] = meta['dominant_class'].map(CLASS_NAMES)
    return meta

metadata_by_state = {state: load_metadata_with_dominant_class(state, path) for state, path in STATE_PATHS.items()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=False)
for ax, state in zip(axes, STATE_PATHS.keys()):
    meta = metadata_by_state[state]
    lon_col = 'center_lon' if 'center_lon' in meta.columns else ('lon' if 'lon' in meta.columns else None)
    lat_col = 'center_lat' if 'center_lat' in meta.columns else ('lat' if 'lat' in meta.columns else None)
    if lon_col is None or lat_col is None:
        raise ValueError(f"Metadata for {state} must include center_lon/center_lat or lon/lat columns")

    for cls in [0, 1, 2, 3]:
        sub = meta[meta['dominant_class'] == cls]
        if len(sub) == 0:
            continue
        ax.scatter(sub[lon_col], sub[lat_col], s=8, alpha=0.6, color=CLASS_COLORS[cls], label=CLASS_NAMES[cls])

    ax.set_title(f'{state}: dominant irrigation class by tile')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.show()

**Interpretation placeholder:**
- Describe class-wise spatial clustering and risk of geographic shortcut learning.

## Section 7 — Sample Visualization
Show representative patches per class (RGB, NDVI, label overlay).

In [ ]:
def find_example_tiles_by_class(state='Colorado', per_class=4):
    out = {cls: [] for cls in range(4)}
    ids = all_tile_ids[state]
    for tid in tqdm(RNG.permutation(ids), desc=f'Finding example tiles ({state})'):
        if all(len(v) >= per_class for v in out.values()):
            break
        _, label = load_tile(STATE_PATHS[state], tid, season='s4')
        for cls in range(4):
            if len(out[cls]) < per_class and np.any(label == cls):
                out[cls].append(tid)
    return out

example_tiles = find_example_tiles_by_class(state='Colorado', per_class=4)
example_tiles

In [ ]:
for cls in range(4):
    tids = example_tiles[cls]
    if len(tids) == 0:
        continue

    fig, axes = plt.subplots(3, len(tids), figsize=(3.5 * len(tids), 8))
    if len(tids) == 1:
        axes = np.array(axes).reshape(3, 1)

    for col, tid in enumerate(tids):
        image, label = load_tile(STATE_PATHS['Colorado'], tid, season='s4')
        rgb = robust_normalize(image[[2, 1, 0]])
        rgb = np.moveaxis(rgb, 0, -1)
        ndvi = image[9]

        axes[0, col].imshow(rgb)
        axes[0, col].set_title(f"Tile {tid:04d}")
        axes[0, col].axis('off')

        axes[1, col].imshow(ndvi, cmap='viridis')
        axes[1, col].axis('off')

        overlay = np.zeros((*label.shape, 4), dtype=float)
        for c in range(4):
            color = CLASS_COLORS[c]
            rgb_color = tuple(int(color.lstrip('#')[i:i+2], 16)/255 for i in (0, 2, 4))
            mask = label == c
            overlay[mask, :3] = rgb_color
            overlay[mask, 3] = 0.35

        axes[2, col].imshow(rgb)
        axes[2, col].imshow(overlay)
        axes[2, col].axis('off')

    axes[0, 0].set_ylabel('RGB (s4)', fontsize=10)
    axes[1, 0].set_ylabel('NDVI median', fontsize=10)
    axes[2, 0].set_ylabel('Label overlay', fontsize=10)
    fig.suptitle(f'Class examples: {CLASS_NAMES[cls]}', y=1.02)
    plt.tight_layout()
    plt.show()

**Interpretation placeholder:**
- Describe visual cues per class and where RGB appears ambiguous relative to NDVI/derived bands.

## Section 8 — Summary and Recommendations

Fill this after running all sections with your concrete numbers.

```markdown
## Summary

**Class distribution:** [fill in based on results]
- Recommended class weights: [computed values]

**Spectral separability:**
- Best bands for flood vs sprinkler: [ranked list]
- RGB provides insufficient separation — spectral input is essential

**Temporal value:**
- [Does temporal help? Which classes benefit most?]

**Label noise:**
- [X]% likely mislabeled irrigated pixels
- [Y]% likely unlabeled irrigated fields
- Recommended NDVI thresholds: high=[value], low=[value]

**OOD difficulty:**
- Most shifted bands: [list]
- Most stable features: [list]
- Expected OOD degradation: [high/moderate/low]

**Recommended next steps:**
1. Add class weights [values] to loss function
2. Move to spectral input (14 bands) — RGB cannot separate flood/sprinkler
3. Apply bidirectional NDVI cleaning with thresholds [values]
4. [Any other data-driven recommendations]
```